# ShadowLM 0.4 — test drive (Colab GPU)

`slm♥` from Lyzr Research Labs · [github](https://github.com/open-gitagent/shadowLM)

Runtime → **Change runtime type → GPU (T4)** first. This walks through the new 0.4 features:
**batteries-included install**, **checkpointing**, **faiss-backed MoRE**, and **APO** (prompt optimization).
If an import fails right after install, do **Runtime → Restart session** once, then re-run.


## 0 · Install — one line, batteries included


In [ ]:
# batteries-included install. -U torchao avoids a stale-version clash with
# Colab's preinstalled torchao (peft needs >=0.16). Restart the session if asked.
!pip install -q -U shadowlm torchao
import shadowlm as slm
print('shadowlm', slm.__version__)


## 1 · Fine-tune with checkpoints + held-out eval
Grab the grounded Lyzr dataset, LoRA-tune a 0.5B model, save a checkpoint every 20 steps, hold out 15% for eval.


In [ ]:
!wget -q https://raw.githubusercontent.com/open-gitagent/shadowLM/main/examples/lyzr_dataset.jsonl
ds = slm.Dataset.from_jsonl('lyzr_dataset.jsonl')
print(len(ds.rows), 'rows ·', ds.format)

model = slm.load('Qwen/Qwen2.5-0.5B-Instruct', accelerator='shadow')
run = model.finetune(ds, method='lora', max_steps=60, save_steps=20, eval_dataset='15%')
print('final loss:', run.loss)
print('eval loss :', run.eval_loss)
print('checkpoints:', [c.label for c in run.checkpoints()])


## 2 · Inference + A/B a checkpoint
Talk to the freshly-tuned model, then load an *earlier* checkpoint and compare — this is what the studio's `ckpt` chips do.


In [ ]:
q = 'What does Lyzr Cloud cost per agent run?'
print('final  :', model.generate(q, max_new_tokens=40, temperature=0.0))

early = slm.load('Qwen/Qwen2.5-0.5B-Instruct', adapter=run.checkpoint_at(20))
print('step 20:', early.generate(q, max_new_tokens=40, temperature=0.0))


## 3 · MoRE — faiss-backed fact memory (near-zero hallucination)
Index the facts behind a faiss `IndexFlatL2` fused into attention. Watch recall flip from made-up to exact.
_(Optional — a bit slower; skip if you're short on time.)_


In [ ]:
mm = slm.load('Qwen/Qwen2.5-0.5B-Instruct')
qa = 'Which Lyzr agent is the marketer?'
print('before:', mm.generate(qa, max_new_tokens=24, temperature=0.0))
# MoRE is slower per step than LoRA — each forward does a CPU faiss lookup,
# so we log every 2 steps and keep it short. Expect a step line within ~30s.
mm.finetune(ds, method='more', max_steps=40, retrieval_k=2,
            retrieval_layers=4, logging_steps=2)
print('after :', mm.generate(qa, max_new_tokens=24, temperature=0.0))


## 4 · APO — optimize the prompt, not the weights (no training)
Same idea, zero gradients: score a system prompt on examples, let the model propose better ones, keep the best.


In [ ]:
base = slm.load('Qwen/Qwen2.5-0.5B-Instruct')
qa = [
  {'question': 'What does Lyzr Cloud cost per agent run?', 'answer': '$0.08'},
  {'question': 'Which Lyzr agent is the marketer?',        'answer': 'Skott'},
  {'question': 'Where is Lyzr headquartered?',             'answer': 'Jersey City'},
  {'question': 'What is Lyzr Architect?',                  'answer': 'no-code agent builder'},
]
apo = slm.optimize_prompt(base, qa, rounds=2, candidates=3)
print('score:', apo.base_score, '->', apo.best_score, apo.sparkline())
print('best prompt:\n', apo.best_prompt)


## 5 · Own the weights
Export the adapter — it's yours; nothing left your machine.


In [ ]:
path = model.save('lyzr-shadow/', fmt='adapter')
print('saved →', path)


---
**More:** the studio (`shadowlm serve`), the remote backend (`backend='remote'`), and production multi-GPU RL
(`backend='verl'`) aren't shown here. Install + open the studio locally with:
`curl -fsSL https://install.shadowlm.sh | sh`
